# Fine-tune CRAFT for Hebrew letter detection (Phase 2)

Generates synthetic Hebrew lines + CRAFT region/affinity heatmaps from the fonts, loads CRAFT
initialized from `craft_mlt_25k`, and fine-tunes it (MSE on the two heatmaps) so its region map
gives **sharp per-letter peaks** instead of merging adjacent Hebrew letters.

**Setup:** Internet ON, GPU ON. Uses `fonts.zip` from Drive (same link as the classifier notebook).
Output: `craft_hebrew.pth` — download it into the repo's `models/` and point `run_craft.py` at it.

In [ ]:
# ── 0. CRAFT code + weights + fonts ────────────────────────────────────
!pip install -q gdown
import os, re, sys, glob, zipfile, pathlib
!git clone -q https://github.com/clovaai/CRAFT-pytorch
# patch vgg16_bn for modern torchvision (model_urls removed)
p = pathlib.Path('CRAFT-pytorch/basenet/vgg16_bn.py'); s = p.read_text()
s = s.replace('from torchvision.models.vgg import model_urls', '')
s = s.replace("model_urls['vgg16_bn'] = model_urls['vgg16_bn'].replace('https://', 'http://')", '')
s = s.replace('models.vgg16_bn(pretrained=pretrained)', 'models.vgg16_bn(weights=None)')
p.write_text(s); sys.path.insert(0, 'CRAFT-pytorch')
# weights (HF mirror; official Google Drive link is quota-blocked)
!curl -fsSL -o craft_mlt_25k.pth https://huggingface.co/boomb0om/dataset-filters/resolve/main/craft_mlt_25k.pth
# fonts
import gdown
FONTS_URL='https://drive.google.com/file/d/1DnF8UGjGQlZCUPx2F02gIg4nNiqZF5Xh/view?usp=sharing'
fid=next(g for g in re.search(r'/d/([\w-]+)|[?&]id=([\w-]+)',FONTS_URL).groups() if g)
gdown.download(f'https://drive.google.com/uc?id={fid}','fonts.zip',quiet=True); zipfile.ZipFile('fonts.zip').extractall('fonts')
FONTS=sorted(glob.glob('fonts/**/*.ttf',recursive=True)); print('fonts:',len(FONTS))


In [ ]:
# ── 1. params ──────────────────────────────────────────────────────────
import numpy as np, cv2, torch
from PIL import Image, ImageDraw, ImageFont
ALPHABET=list('אבגדהוזחטיכלמנסעפצקרשת')+list('ךםןףץ')
IMG_H, IMG_W = 96, 384          # input; CRAFT output is half: 48 x 192
OUT_H, OUT_W = IMG_H//2, IMG_W//2
RENDER_H=56; SEED=1234; rng=np.random.default_rng(SEED)
STEPS, BATCH, LR = 1500, 8, 1e-4   # ~training steps


In [ ]:
# ── 2. synthetic line + heatmap generation ─────────────────────────────
def render_glyph(font,ch):
    asc,desc=font.getmetrics(); box=font.getbbox(ch)
    if box is None: return None
    w=max(box[2]-box[0],1)+8; h=asc+desc+8
    im=Image.new('L',(w,h),0); ImageDraw.Draw(im).text((4-box[0],4),ch,font=font,fill=255)
    a=np.asarray(im); ys,xs=np.where(a>10)
    return None if len(xs)==0 else a[ys.min():ys.max()+1,xs.min():xs.max()+1]

def gaussian_template(size=64):
    ax=np.linspace(-1,1,size); xx,yy=np.meshgrid(ax,ax)
    g=np.exp(-(xx**2+yy**2)/(2*0.35**2)); return (g/g.max()).astype(np.float32)
TMPL=gaussian_template()

def place_gauss(heat,box):
    x0,y0,x1,y1=[int(v) for v in box]; w,h=max(x1-x0,1),max(y1-y0,1)
    g=cv2.resize(TMPL,(w,h)); H,W=heat.shape
    x0=max(x0,0);y0=max(y0,0);x1=min(x0+w,W);y1=min(y0+h,H)
    if x1<=x0 or y1<=y0: return
    heat[y0:y1,x0:x1]=np.maximum(heat[y0:y1,x0:x1],g[:y1-y0,:x1-x0])

def make_sample():
    fp=FONTS[int(rng.integers(len(FONTS)))]
    try: font=ImageFont.truetype(fp,RENDER_H)
    except: return make_sample()
    img=np.full((IMG_H,IMG_W),255,np.uint8); boxes=[]
    x=int(rng.integers(8,28))
    while x < IMG_W-30:
        ch=ALPHABET[int(rng.integers(len(ALPHABET)))]; g=render_glyph(font,ch)
        if g is None: continue
        gh,gw=g.shape
        if gh>IMG_H-8 or x+gw>IMG_W-8: break
        y=int((IMG_H-gh)//2+rng.integers(-6,7)); y=max(0,min(y,IMG_H-gh))
        img[y:y+gh,x:x+gw]=np.minimum(img[y:y+gh,x:x+gw],255-g)  # black ink on white
        boxes.append((x,y,x+gw,y+gh)); x+=gw+int(rng.integers(6,26))
    # light augmentation
    if rng.random()<0.5: img=cv2.GaussianBlur(img,(0,0),rng.uniform(0.3,1.0))
    img=np.clip(img.astype(np.float32)+rng.normal(0,rng.uniform(0,10),img.shape),0,255).astype(np.uint8)
    region=np.zeros((OUT_H,OUT_W),np.float32); affin=np.zeros((OUT_H,OUT_W),np.float32)
    hb=[(x0/2,y0/2,x1/2,y1/2) for (x0,y0,x1,y1) in boxes]
    for b in hb: place_gauss(region,b)
    for a,b in zip(hb,hb[1:]):
        cx0=(a[0]+a[2])/2; cx1=(b[0]+b[2])/2; cy=((a[1]+a[3])/2+(b[1]+b[3])/2)/2
        bh=((a[3]-a[1])+(b[3]-b[1]))/4
        place_gauss(affin,[min(cx0,cx1),cy-bh,max(cx0,cx1),cy+bh])
    return img, region, affin

# normalization identical to CRAFT inference
import imgproc
def make_batch(bs):
    imgs,regs,affs=[],[],[]
    for _ in range(bs):
        im,r,a=make_sample(); imgs.append(imgproc.normalizeMeanVariance(cv2.cvtColor(im,cv2.COLOR_GRAY2RGB)))
        regs.append(r); affs.append(a)
    x=torch.from_numpy(np.stack(imgs)).permute(0,3,1,2).float()
    return x, torch.from_numpy(np.stack(regs)), torch.from_numpy(np.stack(affs))
im,r,a=make_sample(); print('sample img',im.shape,'region',r.shape,'boxes peak',r.max())


In [ ]:
# ── 3. load CRAFT, init from craft_mlt_25k ─────────────────────────────
from collections import OrderedDict
from craft import CRAFT
def copy_sd(sd):
    return OrderedDict((k[7:],v) for k,v in sd.items()) if list(sd)[0].startswith('module') else sd
dev='cuda' if torch.cuda.is_available() else 'cpu'
net=CRAFT(); net.load_state_dict(copy_sd(torch.load('craft_mlt_25k.pth',map_location='cpu'))); net=net.to(dev).train()
print('CRAFT loaded on',dev)


In [ ]:
# ── 4. fine-tune (MSE on region + affinity) ────────────────────────────
from tqdm.auto import trange
opt=torch.optim.Adam(net.parameters(),lr=LR); mse=torch.nn.MSELoss()
best=1e9
for step in trange(STEPS):
    x,rg,af=make_batch(BATCH); x=x.to(dev); rg=rg.to(dev); af=af.to(dev)
    y,_=net(x)                       # y: [B,OUT_H,OUT_W,2]
    pr=y[:,:,:,0]; pa=y[:,:,:,1]
    loss=mse(pr,rg)+mse(pa,af)
    opt.zero_grad(); loss.backward(); opt.step()
    if (step+1)%100==0:
        l=loss.item(); print(f'step {step+1}/{STEPS} loss={l:.4f}')
        if l<best: best=l; torch.save(net.state_dict(),'craft_hebrew.pth')
torch.save(net.state_dict(),'craft_hebrew.pth'); print('saved craft_hebrew.pth (best loss %.4f)'%best)


In [ ]:
# ── 5. sanity: region map before/after on a fresh sample ───────────────
import matplotlib.pyplot as plt
net.eval()
im,r,a=make_sample(); xb,_,_=make_batch(1)
xb=torch.from_numpy(imgproc.normalizeMeanVariance(cv2.cvtColor(im,cv2.COLOR_GRAY2RGB))).permute(2,0,1)[None].float().to(dev)
with torch.no_grad(): y,_=net(xb)
reg=y[0,:,:,0].cpu().numpy()
fig,ax=plt.subplots(3,1,figsize=(10,5))
ax[0].imshow(im,cmap='gray'); ax[0].set_title('input'); ax[0].axis('off')
ax[1].imshow(r,cmap='jet'); ax[1].set_title('target region (GT)'); ax[1].axis('off')
ax[2].imshow(reg,cmap='jet'); ax[2].set_title('predicted region (fine-tuned)'); ax[2].axis('off')
plt.tight_layout(); plt.show()


In [ ]:
# ── 6. download link ───────────────────────────────────────────────────
import os
from IPython.display import FileLink, display
os.chdir('/kaggle/working')   # craft_hebrew.pth was already saved here during training
print('size MB:', round(os.path.getsize('craft_hebrew.pth')/1e6, 1))
display(FileLink('craft_hebrew.pth'))
